In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys, os, re, json, math, random
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/TLM_V03')
assert DRIVE_ROOT.exists(), f"{DRIVE_ROOT} not found -- upload the project to Drive first"
sys.path.insert(0, str(DRIVE_ROOT))

%pip install -q transformers

from src.model_config import ModelConfig
from src.tokenizer import TLM_Tokenizer
from src.data_set import make_sft_sr, apply_sr, split_lines_by_newline, join_lines
from src.data_set.bridging_corruption import make_bridge_sr
from src.data_loaders import PrecomputedBridgingDataset, collate_sft
from src.training import make_scheduler

import torch
from torch.utils.data import DataLoader
from transformers import T5Config, T5ForConditionalGeneration
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  PyTorch: {torch.__version__}")

In [ ]:
BRIDGING_DIR   = DRIVE_ROOT / "data" / "bridging"
TRAIN_IDS_PATH = BRIDGING_DIR / "bridging_train_ids.jsonl"
VAL_IDS_PATH   = BRIDGING_DIR / "bridging_val_ids.jsonl"
VAL_TEXT_PATH  = BRIDGING_DIR / "bridging_val.jsonl"

TOKENIZER_PATH = str(DRIVE_ROOT / "src" / "tokenizer" / "tokenizer_tlm_small.json")

CODET5_PRETRAIN_CKPT_DIR = DRIVE_ROOT / "data" / "checkpoints" / "codet5_arch_pretrain_small"

RUN_NAME = "codet5_small_bridging"
BRIDGING_CKPT_DIR = DRIVE_ROOT / "data" / "checkpoints" / RUN_NAME
BRIDGING_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"CODET5_PRETRAIN_CKPT_DIR={CODET5_PRETRAIN_CKPT_DIR}  BRIDGING_CKPT_DIR={BRIDGING_CKPT_DIR}")
for p in (TRAIN_IDS_PATH, VAL_IDS_PATH, VAL_TEXT_PATH):
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")

In [ ]:
tlm_config = ModelConfig()
PAD_ID = tlm_config.special_token_ids["pad"]
BOS_ID = tlm_config.special_token_ids["bos"]
EOS_ID = tlm_config.special_token_ids["eos"]

tokenizer = TLM_Tokenizer(vocab_size=tlm_config.vocab_size)
tokenizer.load(TOKENIZER_PATH)
VOCAB_SIZE = tokenizer.tokenizer.get_vocab_size()
print(f"Tokenizer loaded from {TOKENIZER_PATH}  |  vocab_size={VOCAB_SIZE}  pad={PAD_ID}  bos={BOS_ID}  eos={EOS_ID}")

t5_config = T5Config.from_pretrained("Salesforce/codet5-small")
t5_config.vocab_size = VOCAB_SIZE
t5_config.d_model = 256
t5_config.d_ff = 1024
t5_config.num_layers = 4
t5_config.num_decoder_layers = 4
t5_config.num_heads = 4
t5_config.d_kv=64
t5_config.pad_token_id = PAD_ID
t5_config.eos_token_id = EOS_ID
t5_config.decoder_start_token_id = BOS_ID

model = T5ForConditionalGeneration(t5_config).to(device)

_ckpts = sorted(CODET5_PRETRAIN_CKPT_DIR.glob("codet5_arch_pretrain_epoch_*_val*.pt"))
assert _ckpts, f"No checkpoints found in {CODET5_PRETRAIN_CKPT_DIR}"

def _val_loss_from_name(p):
    m = re.search(r"val([0-9.]+)\.pt$", p.name)
    return float(m.group(1)) if m else float("inf")

PRETRAIN_CKPT_PATH = min(_ckpts, key=_val_loss_from_name)
print(f"Found {len(_ckpts)} checkpoint(s) in {CODET5_PRETRAIN_CKPT_DIR}")
print(f"Using: {PRETRAIN_CKPT_PATH.name}")

_state = torch.load(PRETRAIN_CKPT_PATH, map_location=device)
model.load_state_dict(_state["model"])
print(f"Loaded pretrain checkpoint: epoch {_state.get('epoch')}  val_loss {_state.get('val_loss')}")

n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params/1e6:.3f}M")

model.gradient_checkpointing_enable()
model.config.use_cache = False

In [ ]:
train_ds = PrecomputedBridgingDataset(TRAIN_IDS_PATH)
val_ds   = PrecomputedBridgingDataset(VAL_IDS_PATH)
print(f"train: {len(train_ds):,} samples")
print(f"val:   {len(val_ds):,} samples")

BATCH_SIZE       = 32
GRAD_ACCUM_STEPS = 1
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
_PIN        = torch.cuda.is_available()
_collate    = lambda b: collate_sft(b, PAD_ID)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=_collate, num_workers=NUM_WORKERS, pin_memory=_PIN)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=_collate, num_workers=NUM_WORKERS, pin_memory=_PIN)
print(f"train batches: {len(train_loader)}  |  val batches: {len(val_loader)}  |  "
      f"effective batch size: {EFFECTIVE_BATCH_SIZE}  |  device: {device}")

In [ ]:
def hf_step_batch(model, batch, pad_id, device, amp_dtype=None):
    enc_ids  = batch["enc_ids"].to(device)
    enc_attn = batch["enc_attn"].to(device)
    dec_ids  = batch["dec_ids"].to(device)

    labels = dec_ids[:, 1:].clone()
    labels[labels == pad_id] = -100

    if amp_dtype is not None:
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            out = model(input_ids=enc_ids, attention_mask=enc_attn, labels=labels)
    else:
        out = model(input_ids=enc_ids, attention_mask=enc_attn, labels=labels)
    return out.loss

@torch.no_grad()
def hf_validate(model, loader, pad_id, device, amp_dtype=None, log_every=200):
    model.eval()
    total_loss, n_batches = 0.0, 0
    for i, batch in enumerate(loader):
        loss = hf_step_batch(model, batch, pad_id, device, amp_dtype=amp_dtype)
        total_loss += float(loss.detach())
        n_batches += 1
        if (i + 1) % log_every == 0:
            print(f"    [val] {i+1}/{len(loader)}", flush=True)
    return total_loss / max(n_batches, 1)

def hf_run_epoch(model, loader, optimizer, scheduler, pad_id, device,
                 grad_accum_steps=1, amp_dtype=None, grad_clip=1.0, log_every=200):
    model.train()
    total_loss, n_micro = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for i, batch in enumerate(loader):
        loss = hf_step_batch(model, batch, pad_id, device, amp_dtype=amp_dtype)
        (loss / grad_accum_steps).backward()
        total_loss += float(loss.detach())
        n_micro += 1

        if n_micro % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if (i + 1) % log_every == 0:
            print(f"    step {i+1}/{len(loader)}  loss={total_loss/n_micro:.4f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}", flush=True)

    if n_micro % grad_accum_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    return total_loss / max(n_micro, 1)

EPOCHS       = 4
LR           = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
GRAD_CLIP    = 1.0
PATIENCE     = 2
USE_AMP      = False
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_available()
                               and torch.cuda.is_bf16_supported()) else None

steps_per_epoch = len(train_loader) // GRAD_ACCUM_STEPS
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = make_scheduler(optimizer, warmup_steps, total_steps)

print(f"epochs={EPOCHS}  steps/epoch={steps_per_epoch}  total={total_steps}  "
      f"warmup={warmup_steps}  lr={LR}  effective_batch={EFFECTIVE_BATCH_SIZE}")

In [ ]:
RESUME_PATH = BRIDGING_CKPT_DIR / "last.pt"

best_val = float("inf")
best_ckpt_path = None
bad_epochs = 0
history = []
start_epoch = 1

if RESUME_PATH.exists():
    print(f"Resuming from {RESUME_PATH}")
    _ckpt = torch.load(RESUME_PATH, map_location=device)
    model.load_state_dict(_ckpt["model"])
    optimizer.load_state_dict(_ckpt["optimizer"])
    scheduler.load_state_dict(_ckpt["scheduler"])
    start_epoch = _ckpt["epoch"] + 1
    best_val = _ckpt["best_val"]
    bad_epochs = _ckpt["bad_epochs"]
    best_ckpt_path = Path(_ckpt["best_ckpt_path"]) if _ckpt.get("best_ckpt_path") else None
    history = _ckpt["history"]
    print(f"  resuming at epoch {start_epoch}  |  best_val so far {best_val:.4f}  |  bad_epochs {bad_epochs}/{PATIENCE}")
else:
    print("No last.pt found -- starting fresh bridging run")

for epoch in range(start_epoch, EPOCHS + 1):
    print(f"\n=== Epoch {epoch}/{EPOCHS} ===")
    tr_loss = hf_run_epoch(model, train_loader, optimizer, scheduler, PAD_ID, device,
                           grad_accum_steps=GRAD_ACCUM_STEPS, amp_dtype=AMP_DTYPE, grad_clip=GRAD_CLIP)
    va_loss = hf_validate(model, val_loader, PAD_ID, device, amp_dtype=AMP_DTYPE)
    print(f"epoch {epoch}: train_loss {tr_loss:.4f}  val_loss {va_loss:.4f}")
    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss})

    if va_loss < best_val:
        best_val = va_loss
        bad_epochs = 0
        best_ckpt_path = BRIDGING_CKPT_DIR / f"{RUN_NAME}_epoch_{epoch}_val{va_loss:.4f}.pt"
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_loss": va_loss,
                    "t5_config": t5_config.to_dict()}, best_ckpt_path)
        print(f"  saved best -> {best_ckpt_path.name}")
    else:
        bad_epochs += 1
        print(f"  no improvement ({bad_epochs}/{PATIENCE})")

    torch.save({
        "model": model.state_dict(), "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(), "epoch": epoch,
        "best_val": best_val, "bad_epochs": bad_epochs,
        "best_ckpt_path": str(best_ckpt_path) if best_ckpt_path else None,
        "history": history,
    }, RESUME_PATH)

    if bad_epochs >= PATIENCE:
        print("early stopping.")
        break

print(f"\nbest val_loss: {best_val:.4f}")


In [ ]:
import matplotlib.pyplot as plt

history_path = BRIDGING_CKPT_DIR / "training_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"history saved -> {history_path}")

epochs_x   = [h["epoch"] for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss   = [h["val_loss"] for h in history]
best_ep    = epochs_x[val_loss.index(min(val_loss))]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(epochs_x, train_loss, marker="o", label="train loss")
ax.plot(epochs_x, val_loss, marker="o", label="val loss")
ax.axvline(x=best_ep, color="green", linestyle="--", alpha=0.5, label=f"best (ep {best_ep})")
ax.set_xlabel("epoch"); ax.set_ylabel("loss")
ax.set_title(f"{RUN_NAME} ({n_params/1e6:.1f}M params) -- bridging loss curves")
ax.legend()
plt.tight_layout()
plt.savefig(BRIDGING_CKPT_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
assert best_ckpt_path is not None, "no checkpoint saved -- did training run?"
_best_state = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(_best_state["model"])
model.eval()
model.config.use_cache = True
print(f"loaded best: {best_ckpt_path.name}  val_loss={_best_state['val_loss']:.4f}")

In [ ]:
from collections import defaultdict

MAX_NEW_TOKENS = 512
_NL = tlm_config.special_token_ids["newline"]

@torch.no_grad()
def hf_greedy_decode(model, enc_ids, device, max_new_tokens=MAX_NEW_TOKENS):
    input_ids = torch.tensor([enc_ids], dtype=torch.long, device=device)
    attn = torch.ones_like(input_ids)
    out = model.generate(
        input_ids=input_ids, attention_mask=attn,
        max_new_tokens=max_new_tokens, num_beams=1, do_sample=False,
        pad_token_id=PAD_ID, eos_token_id=EOS_ID, decoder_start_token_id=BOS_ID,
    )
    return out[0].tolist()

@torch.no_grad()
def evaluate_bridging_applied_ok(model, val_text_path, tokenizer, cfg, n_per_type=100, seed=0):
    by_type = defaultdict(list)
    with open(val_text_path, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            by_type[d["type"]].append(d)
    rng = random.Random(seed)
    results = {}
    for ttype, recs in sorted(by_type.items()):
        sample = rng.sample(recs, min(n_per_type, len(recs)))
        build_fn = make_bridge_sr if ttype == "rename" else make_sft_sr
        ok = trunc = total = 0
        for d in sample:
            res = build_fn(d, tokenizer, cfg)
            if res is None:
                continue
            enc_ids, _, _ = res
            before_ids = tokenizer.encode(d["before"])
            after_ids = tokenizer.encode(d["after"])
            gold_after = join_lines(split_lines_by_newline(after_ids, _NL), _NL)
            pred_dec = hf_greedy_decode(model, enc_ids, device)
            pred_after = apply_sr(before_ids, pred_dec, cfg)
            total += 1
            if pred_dec[-1] != EOS_ID:
                trunc += 1
            if pred_after is not None and pred_after == gold_after:
                ok += 1
        results[ttype] = (ok, total, trunc)
    return results

res = evaluate_bridging_applied_ok(model, VAL_TEXT_PATH, tokenizer, tlm_config, n_per_type=100)
print(f"{'task':18}{'applied_ok':>12}{'truncated':>11}")
oks = tots = 0
for t, (ok, tot, trunc) in res.items():
    oks += ok
    tots += tot
    print(f"{t:18}{ok:4d}/{tot:<4d} {ok/max(tot, 1)*100:5.1f}%   {trunc:3d}/{tot:<3d}")
print(f"{'OVERALL':18}{oks:4d}/{tots:<4d} {oks/max(tots, 1)*100:5.1f}%")
